# TrashTrack — Training Pipeline (Colab)
**MSDS 498 Capstone · Group 4**

This notebook runs the full training + evaluation pipeline on a free Colab T4 GPU.

**What it does:**
1. Clones the repo & installs deps
2. Mounts Google Drive (for dataset storage + saving weights)
3. Downloads & converts all three datasets (TACO, UAVVaste, RoLID-11K)
4. Merges into a single training set
5. **Run A** — train on TACO only (baseline)
6. **Run B** — train on merged (all 3 datasets)
7. Cross-dataset evaluation (ST-04 generalization benchmark)
8. Saves best weights + results to Drive

**Before you start:** Make sure you're on a **T4 GPU** runtime.
`Runtime → Change runtime type → T4 GPU`


## 1. Setup

In [ ]:
# Verify GPU is available
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
# Clone the TrashTrack repo
!git clone https://github.com/<YOUR-USERNAME>/trashtrack.git
%cd trashtrack


In [ ]:
# Install dependencies
!pip install -q ultralytics pillow numpy pydantic


In [ ]:
# Mount Google Drive (for persistent storage across sessions)
from google.colab import drive
drive.mount('/content/drive')

# Create a persistent folder on Drive
DRIVE_BASE = '/content/drive/MyDrive/trashtrack_data'
!mkdir -p {DRIVE_BASE}/datasets
!mkdir -p {DRIVE_BASE}/weights
!mkdir -p {DRIVE_BASE}/results
print(f"Drive storage at: {DRIVE_BASE}")


## 2. Download Datasets

⏱ This takes ~10-15 minutes the first time. Once done, datasets live on Drive
so you skip this step in future sessions.

**Run this section only once** — then comment it out for subsequent sessions.


In [ ]:
# ── Check if datasets already exist on Drive ──
import os

datasets_ready = all(
    os.path.exists(f"{DRIVE_BASE}/datasets/{ds}/images/train")
    for ds in ["taco", "uavvaste", "rolid11k"]
)

if datasets_ready:
    print("✅ Datasets already on Drive — skipping download. Symlinking...")
    !ln -sf {DRIVE_BASE}/datasets datasets
else:
    print("⬇ Datasets not found — will download and convert.")
    print("  This only needs to happen once.")


In [ ]:
# ── 2a. TACO (street-level, ~1,500 images) ──
# Skip if datasets_ready is True

if not datasets_ready:
    %cd /content/trashtrack
    !mkdir -p data

    # Clone TACO repo
    !git clone --depth 1 https://github.com/pedropro/TACO.git data/taco_raw

    # Download TACO images (uses their download script)
    %cd data/taco_raw
    !pip install -q pycocotools
    !python download.py
    %cd /content/trashtrack

    # Convert: 60 classes → single 'litter', COCO → YOLO format
    !python scripts/convert_taco.py \
        --annotations data/taco_raw/data/annotations.json \
        --images-dir data/taco_raw/data \
        --output datasets/taco


In [ ]:
# ── 2b. UAVVaste (drone/aerial, ~772 images) ──

if not datasets_ready:
    !git clone --depth 1 https://github.com/UAVVaste/UAVVaste.git data/uavvaste_raw

    !python scripts/convert_uavvaste.py \
        --annotations data/uavvaste_raw/annotations.json \
        --images-dir data/uavvaste_raw/images \
        --output datasets/uavvaste


In [ ]:
# ── 2c. RoLID-11K (dashcam, ~11,000 images) ──

if not datasets_ready:
    !git clone --depth 1 https://github.com/xq141839/RoLID-11K.git data/rolid11k_raw

    # Check if it has train/val subdirs or is flat
    import os
    has_splits = os.path.isdir("data/rolid11k_raw/images/train")

    if has_splits:
        !python scripts/organize_rolid.py \
            --images-dir data/rolid11k_raw/images \
            --labels-dir data/rolid11k_raw/labels \
            --output datasets/rolid11k \
            --already-split
    else:
        !python scripts/organize_rolid.py \
            --images-dir data/rolid11k_raw/images \
            --labels-dir data/rolid11k_raw/labels \
            --output datasets/rolid11k


In [ ]:
# ── 2d. Merge all three ──

if not datasets_ready:
    !python scripts/merge_datasets.py --output datasets/merged

    # Copy converted datasets to Drive for next time
    print("\n📁 Copying datasets to Drive (one-time)...")
    !cp -r datasets/* {DRIVE_BASE}/datasets/
    print("✅ Datasets saved to Drive.")


## 3. Verify Datasets

In [ ]:
# Sanity check — count images and labels per dataset
import os

def count_files(path, ext):
    if not os.path.exists(path):
        return 0
    return len([f for f in os.listdir(path) if f.endswith(ext)])

# Symlink to Drive datasets if not already linked
if not os.path.exists("datasets/taco"):
    !ln -sf {DRIVE_BASE}/datasets datasets

print("Dataset Summary:")
print("=" * 55)
for ds in ["taco", "uavvaste", "rolid11k", "merged"]:
    for split in ["train", "val"]:
        imgs = count_files(f"datasets/{ds}/images/{split}", ".jpg") + \
               count_files(f"datasets/{ds}/images/{split}", ".png")
        lbls = count_files(f"datasets/{ds}/labels/{split}", ".txt")
        if imgs > 0:
            print(f"  {ds:12s} {split:5s}  {imgs:6d} images  {lbls:6d} labels")
print("=" * 55)


In [ ]:
# Spot-check: view a random label file
import random, glob

label_files = glob.glob("datasets/taco/labels/train/*.txt")
if label_files:
    sample = random.choice(label_files)
    print(f"Sample label: {sample}")
    print(open(sample).read())
    print("\nFormat: class_id cx cy w h (normalized)")
    print("class_id 0 = litter (all 60 TACO classes collapsed)")


## 4. Run A — TACO-Only Baseline

Train on TACO alone. This establishes the single-dataset baseline
to compare against merged training.

⏱ ~3-5 hours on T4 for 100 epochs.


In [ ]:
# Run A: Train on TACO only
!yolo detect train \
    data=configs/taco.yaml \
    model=yolov8s.pt \
    epochs=100 \
    imgsz=640 \
    batch=16 \
    patience=20 \
    name=run_a_taco_only \
    save=True \
    plots=True


In [ ]:
# Evaluate Run A on TACO val
!yolo detect val \
    data=configs/taco.yaml \
    model=runs/detect/run_a_taco_only/weights/best.pt \
    name=run_a_eval_taco \
    plots=True

print("\n🎯 Target: mAP@0.5 ≥ 0.45")


In [ ]:
# Save Run A weights to Drive
!cp runs/detect/run_a_taco_only/weights/best.pt {DRIVE_BASE}/weights/run_a_best.pt
print("✅ Run A weights saved to Drive")


## 5. Run B — Merged Training (All 3 Datasets)

Train on TACO + UAVVaste + RoLID-11K combined.
Then evaluate on each dataset separately for the generalization benchmark.

⏱ ~5-8 hours on T4 for 100 epochs (larger dataset).


In [ ]:
# Run B: Train on merged dataset
!yolo detect train \
    data=configs/merged.yaml \
    model=yolov8s.pt \
    epochs=100 \
    imgsz=640 \
    batch=16 \
    patience=20 \
    name=run_b_merged \
    save=True \
    plots=True


In [ ]:
# Evaluate Run B on each dataset separately (ST-04)
print("=" * 60)
print("  ST-04: Cross-Dataset Generalization Benchmark")
print("=" * 60)

for ds, config, target in [
    ("TACO", "configs/taco.yaml", 0.45),
    ("UAVVaste", "configs/uavvaste.yaml", 0.35),
    ("RoLID-11K", "configs/rolid11k.yaml", 0.30),
]:
    print(f"\n{'─'*40}")
    print(f"  {ds}  (target mAP@0.5 ≥ {target})")
    print(f"{'─'*40}")

    !yolo detect val \
        data={config} \
        model=runs/detect/run_b_merged/weights/best.pt \
        name=run_b_eval_{ds.lower().replace('-','')} \
        plots=True


In [ ]:
# Save Run B weights to Drive
!cp runs/detect/run_b_merged/weights/best.pt {DRIVE_BASE}/weights/run_b_best.pt
print("✅ Run B weights saved to Drive")


## 6. Compare Results

Side-by-side comparison: did merging help generalization?


In [ ]:
# Parse and compare results
import pandas as pd
from pathlib import Path
import csv

def read_results(run_dir):
    """Read the last line of results.csv from a YOLO training run."""
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        return None
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    return df.iloc[-1] if len(df) > 0 else None

# Gather metrics
results = []

# Run A
r = read_results("runs/detect/run_a_taco_only")
if r is not None:
    results.append({
        "Run": "A (TACO only)",
        "Eval Dataset": "TACO",
        "mAP@0.5": r.get("metrics/mAP50(B)", "N/A"),
        "mAP@0.5:0.95": r.get("metrics/mAP50-95(B)", "N/A"),
        "Precision": r.get("metrics/precision(B)", "N/A"),
        "Recall": r.get("metrics/recall(B)", "N/A"),
    })

# Run B — we need to check the eval runs
for ds_name, eval_dir in [
    ("TACO", "run_b_eval_taco"),
    ("UAVVaste", "run_b_eval_uavvaste"),
    ("RoLID-11K", "run_b_eval_rolid11k"),
]:
    eval_path = f"runs/detect/{eval_dir}"
    if Path(eval_path).exists():
        # Val results are printed but may not have CSV — use the training results
        results.append({
            "Run": "B (Merged)",
            "Eval Dataset": ds_name,
            "mAP@0.5": "check output above",
            "mAP@0.5:0.95": "check output above",
        })

if results:
    df = pd.DataFrame(results)
    print("\n📊 Results Comparison")
    print("=" * 70)
    print(df.to_string(index=False))

# Thresholds
print("\n\n🎯 Success Thresholds (professor feedback):")
print("─" * 40)
for ds, t in [("TACO", 0.45), ("UAVVaste", 0.35), ("RoLID-11K", 0.30)]:
    print(f"  {ds:12s}  mAP@0.5 ≥ {t}")
print(f"  {'Precision':12s}  @ conf≥0.50 ≥ 0.70")
print(f"  {'Recall':12s}  @ conf≥0.25 ≥ 0.50")


## 7. Save Everything to Drive

Copy training plots, results, and weights so they persist.


In [ ]:
# Save all results to Drive
import shutil

# Copy entire runs directory
!cp -r runs/detect {DRIVE_BASE}/results/

# List saved weights
print("\n📁 Saved to Drive:")
!ls -la {DRIVE_BASE}/weights/
print()
!ls {DRIVE_BASE}/results/
print("\n✅ All results saved. You can close this Colab session.")


## 8. Use the Trained Model in Your App

Download the best weights and point the TrashTrack app at them:

```bash
# On your local machine
cp /path/to/drive/trashtrack_data/weights/run_b_best.pt trashtrack/weights/best.pt

# Start the API with the fine-tuned model
TT_MODEL_WEIGHTS=weights/best.pt uvicorn trashtrack.api:app --port 8000

# Start the React frontend
cd frontend && npm run dev
```

Now when you upload a street photo, it uses your fine-tuned model
instead of the stock YOLOv8n — and actually detects litter.
